<a href="https://colab.research.google.com/github/andrewbeyou88/uXUbYJ34sxDN11/blob/main/microwakeword/notebooks/basic_training_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Training a microWakeWord Model

This notebook steps you through training a basic microWakeWord model. It is intended as a **starting point** for advanced users. You should use Python 3.10.

**The model generated will most likely not be usable for everyday use; it may be difficult to trigger or falsely activates too frequently. You will most likely have to experiment with many different settings to obtain a decent model!**

In the comment at the start of certain blocks, I note some specific settings to consider modifying.

This runs on Google Colab, but is extremely slow compared to training on a local GPU. If you must use Colab, be sure to Change the runtime type to a GPU. Even then, it still slow!

At the end of this notebook, you will be able to download a tflite file. To use this in ESPHome, you need to write a model manifest JSON file. See the [ESPHome documentation](https://esphome.io/components/micro_wake_word) for the details and the [model repo](https://github.com/esphome/micro-wake-word-models/tree/main/models/v2) for examples.

In [ ]:
# Installs microWakeWord. Be sure to restart the session after this is finished.
import platform

if platform.system() == "Darwin":
    # `pymicro-features` is installed from a fork to support building on macOS
    !pip install 'git+https://github.com/puddly/pymicro-features@puddly/minimum-cpp-version'

# `audio-metadata` is installed from a fork to unpin `attrs` from a version that breaks Jupyter
!pip install 'git+https://github.com/whatsnowplaying/audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f'

!git clone https://github.com/kahrendt/microWakeWord
!pip install -e ./microWakeWord

In [ ]:
# Install Piper Sample Generator and download voice models
!pip install piper-sample-generator
!mkdir -p voices
!wget -O voices/en_US-lessac-medium.onnx 'https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/lessac/medium/en_US-lessac-medium.onnx?download=true'
!wget -O voices/en_US-lessac-medium.onnx.json 'https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/lessac/medium/en_US-lessac-medium.onnx.json?download=true'

In [ ]:
# Create missing directory structure and init files for piper_train compatibility
!mkdir -p /usr/local/lib/python3.12/dist-packages/piper_train/vits
!touch /usr/local/lib/python3.12/dist-packages/piper_train/__init__.py
!touch /usr/local/lib/python3.12/dist-packages/piper_train/vits/__init__.py
!touch /usr/local/lib/python3.12/dist-packages/piper_train/vits/commons.py

In [ ]:
# Generate synthetic wake word samples using Piper
!python -m piper_sample_generator \
  --model voices/en_US-lessac-medium.onnx \
  --text "okay piper" \
  --max-samples 500 \
  --output-dir generated_samples

In [ ]:
# Generate features and spectrograms from synthetic data
!python -m microwakeword.generate_features \
  --input-dir generated_samples \
  --output-dir feature_data

In [ ]:
# Train the microWakeWord model
!python -m microwakeword.train \
  --training-data feature_data \
  --output-dir trained_model

In [ ]:
# Convert trained model to TFLite format for ESP32 / ESPHome deployment
!python -m microwakeword.export_tflite \
  --model-dir trained_model \
  --output-file model.tflite